In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from scipy.stats import mannwhitneyu

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif,
    SelectFromModel, RFE, SequentialFeatureSelector
)
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, mutual_info_score
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier

# PACO

from funciones_pipeline import *

from functools import partial

In [2]:
df = pd.read_excel("TARGET.xlsx")

In [3]:
df

,Country,Systemic Banking Crisis (starting date),Currency Crisis,Sovereign Debt Crisis (year),Sovereign Debt Restructuring (year),a
0,NaN,NaN,(year),NaN,NaN,NaN
1,Albania,1994,1997,1990,1992,UNK
2,Algeria,1990,"1988, 1994",NaN,NaN,UNK
3,Angola,NaN,"1991, 1996, 2015",1988,1992,UNK
4,Argentina,"1980, 1989, 1995, 2001","1975, 1981, 1987, 2002, 2013","1982, 2001, 2014","1993, 2005, 2016",UNK
...,...,...,...,...,...,...
212,2015,NaN,13,1,NaN,NaN
213,2016,NaN,4,NaN,NaN,NaN
214,2017,NaN,NaN,2,NaN,NaN
215,Total,151,236,75,41,11


In [4]:
df.Country[54:109]

54                      France
55                       Gabon
56                 Gambia, The
57                     Georgia
58                     Germany
59                       Ghana
60                      Greece
61                     Grenada
62                   Guatemala
63                      Guinea
64               Guinea-Bissau
65        Guyana              
66        Haiti               
67     Honduras               
68      China, P.R.: Hong Kong
69                     Hungary
70                     Iceland
71                       India
72                   Indonesia
73               Iran, I.R. of
74                     Ireland
75                      Israel
76                       Italy
77                     Jamaica
78                       Japan
79                      Jordan
80        Kazakhstan          
81                       Kenya
82                       Korea
83                      Kuwait
84        Kyrgyz Republic     
85      Lao People’s Dem. Rep.
86      

In [5]:
lista_paises = df.Country.to_list()

In [6]:
len(lista_paises)

217

In [7]:
lista_paises

[nan,
 'Albania',
 'Algeria',
 'Angola',
 'Argentina',
 'Armenia',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bhutan              ',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia            ',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Central African Rep.',
 'Chad',
 'Chile',
 'China, P.R.',
 'Colombia',
 'Comoros             ',
 'Congo, Dem. Rep. of',
 'Congo, Rep. of',
 'Costa Rica',
 'Côte d’Ivoire',
 'Croatia',
 'Czech Republic',
 'Cyprus',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Ethiopia            ',
 'Fiji                ',
 'Finland',
 'France',
 'Gabon',
 'Gambia, The',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Grenada',
 'Guatemala',
 'Guinea',
 'Guinea-Bissau',
 'Guyana              ',
 'Haiti               ',
 'Ho

In [8]:
cambiar_nombre = {
    "Gambia, The": "Gambia",
    "China, P.R.: Hong Kong": "Hong Kong",
    "Iran, I.R. of": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Lao People’s Dem. Rep.": "Laos",
    "Macedonia": "North Macedonia",
    "Korea": "South Korea",
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "Central African Rep.": "Central African Republic",
    "China, P.R.": "China",
    "Congo, Dem. Rep. of": "Congo, The Democratic Republic of the",
    "Congo, Rep. of": "Congo",
    "Côte d’Ivoire": "Côte d'Ivoire",
    "Russia": "Russian Federation",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "São Tomé and Principe": "Sao Tome and Principe",
    "Serbia, Republic of": "Serbia",
    "Swaziland": "Eswatini",
    "Turkey": "Türkiye",
    "Yugoslavia, SFR": "Yugoslavia"
}


In [9]:
import pycountry

def normalize_country(name):
    name = str(name).strip()
    return cambiar_nombre.get(name, name)

def get_iso3(name):
    name = normalize_country(name)
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

def get_iso_numeric(name):
    name = normalize_country(name)
    try:
        return int(pycountry.countries.lookup(name).numeric)
    except:
        return None

In [10]:
df_codigo = pd.DataFrame({"country": lista_paises})
df_codigo

,country
0,NaN
1,Albania
2,Algeria
3,Angola
4,Argentina
...,...
212,2015
213,2016
214,2017
215,Total


In [11]:
df_codigo["iso3"] = df_codigo["country"].apply(get_iso3)
df_codigo["iso_numeric"] = df_codigo["country"].apply(get_iso_numeric)

In [12]:
df_codigo

,country,iso3,iso_numeric
0,NaN,None,NaN
1,Albania,ALB,8.0
2,Algeria,DZA,12.0
3,Angola,AGO,24.0
4,Argentina,ARG,32.0
...,...,...,...
212,2015,None,NaN
213,2016,None,NaN
214,2017,None,NaN
215,Total,None,NaN


In [13]:
df_codigo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   country      216 non-null    object 
 1   iso3         164 non-null    object 
 2   iso_numeric  164 non-null    float64
dtypes: float64(1), object(2)
memory usage: 5.2+ KB


In [14]:
paises_nombre = df_codigo.country.str.strip().to_list()
paises_nombre

[nan,
 'Albania',
 'Algeria',
 'Angola',
 'Argentina',
 'Armenia',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bhutan',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Central African Rep.',
 'Chad',
 'Chile',
 'China, P.R.',
 'Colombia',
 'Comoros',
 'Congo, Dem. Rep. of',
 'Congo, Rep. of',
 'Costa Rica',
 'Côte d’Ivoire',
 'Croatia',
 'Czech Republic',
 'Cyprus',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Ethiopia',
 'Fiji',
 'Finland',
 'France',
 'Gabon',
 'Gambia, The',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Grenada',
 'Guatemala',
 'Guinea',
 'Guinea-Bissau',
 'Guyana',
 'Haiti',
 'Honduras',
 'China, P.R.: Hong Kong',
 'Hungary',
 'Iceland',
 'India',
 'Indonesia',
 'Iran, I.R.

In [15]:
df_codigo.loc[df_codigo.iso3.isna()]

,country,iso3,iso_numeric
0,NaN,None,NaN
163,"Yugoslavia, SFR",None,NaN
166,Year,None,NaN
167,1970,None,NaN
168,1971,None,NaN
169,1972,None,NaN
170,1973,None,NaN
171,1974,None,NaN
172,1975,None,NaN
173,1976,None,NaN


In [16]:
iso_mal = df_codigo.loc[df_codigo.iso3.isna()]["country"].to_list()
iso_mal

[nan,
 'Yugoslavia, SFR',
 'Year',
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 'Total',
 '1/ Twin crisis indicates banking crisis in year t and currency crisis during [t-1, t+1]. 2/ Triple crisis indicates banking crisis in year t and currency crisis during [t-1, t+1] and debt crisis during [t-1, t+1].']

In [17]:
iso_mal

[nan,
 'Yugoslavia, SFR',
 'Year',
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 'Total',
 '1/ Twin crisis indicates banking crisis in year t and currency crisis during [t-1, t+1]. 2/ Triple crisis indicates banking crisis in year t and currency crisis during [t-1, t+1] and debt crisis during [t-1, t+1].']

In [18]:
df_codigo.loc[df_codigo["country"] == "Yugoslavia, SFR", "iso3"] = "YUG"
df_codigo.loc[df_codigo["country"] == "Yugoslavia, SFR", "iso_numeric"] = 896

In [19]:
df_codigo.loc[df_codigo["country"] == "Yugoslavia, SFR"]

,country,iso3,iso_numeric
163,"Yugoslavia, SFR",YUG,896.0


In [20]:
df_codigo = df_codigo[1:166]
df_codigo

,country,iso3,iso_numeric
1,Albania,ALB,8.0
2,Algeria,DZA,12.0
3,Angola,AGO,24.0
4,Argentina,ARG,32.0
5,Armenia,ARM,51.0
...,...,...,...
161,Vietnam,VNM,704.0
162,Yemen,YEM,887.0
163,"Yugoslavia, SFR",YUG,896.0
164,Zambia,ZMB,894.0


In [21]:
df2 = pd.read_excel("Datos_paises_despivotados.xlsx",index_col=0)

In [22]:
df2

,Country Name,Country Code,year,Bank nonperforming loans to total gross loans (%),Broad money (% of GDP),Current account balance (% of GDP),Deposit interest rate (%),Domestic credit to private sector (% of GDP),Domestic credit to private sector by banks (% of GDP),Exports of goods and services (% of GDP),...,"Inflation, consumer prices (annual %)",Lending interest rate (%),Net barter terms of trade index (2015 = 100),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Short-term debt (% of total external debt),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,Trade (% of GDP),"Unemployment, total (% of total labor force) (national estimate)"
0,Afghanistan,AFG,1970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.378709,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13922,Zimbabwe,ZWE,2020,NaN,10.002460,2.121236,4.518333,3.642132,3.607706,18.931225,...,557.201817,33.008333,107.1,51.329013,-80.610262,29.5755,18.680976,0.062756,40.178406,NaN
13923,Zimbabwe,ZWE,2021,NaN,10.239997,0.843388,8.059167,4.759522,4.759505,19.759617,...,98.546105,45.476667,96.6,88.552447,-30.318195,26.4874,9.228518,1.148220,44.114559,9.459
13924,Zimbabwe,ZWE,2022,NaN,12.673110,0.748243,34.720000,5.826323,5.826294,23.699220,...,104.705171,131.813333,94.0,374.954363,-38.093651,28.1510,6.084244,0.704319,55.067352,10.001
13925,Zimbabwe,ZWE,2023,NaN,11.204613,0.373211,62.820833,6.477697,6.477673,17.117076,...,NaN,170.293833,99.1,3509.172220,-68.877267,27.0548,19.516072,0.129141,40.291261,9.290


In [23]:
iso_bien_lista = df_codigo.iso3.to_list()

In [24]:
iso_bien_lista

['ALB',
 'DZA',
 'AGO',
 'ARG',
 'ARM',
 'AUS',
 'AUT',
 'AZE',
 'BGD',
 'BRB',
 'BLR',
 'BEL',
 'BLZ',
 'BEN',
 'BTN',
 'BOL',
 'BIH',
 'BWA',
 'BRA',
 'BRN',
 'BGR',
 'BFA',
 'BDI',
 'KHM',
 'CMR',
 'CAN',
 'CPV',
 'CAF',
 'TCD',
 'CHL',
 'CHN',
 'COL',
 'COM',
 'COD',
 'COG',
 'CRI',
 'CIV',
 'HRV',
 'CZE',
 'CYP',
 'DNK',
 'DJI',
 'DMA',
 'DOM',
 'ECU',
 'EGY',
 'SLV',
 'GNQ',
 'ERI',
 'EST',
 'ETH',
 'FJI',
 'FIN',
 'FRA',
 'GAB',
 'GMB',
 'GEO',
 'DEU',
 'GHA',
 'GRC',
 'GRD',
 'GTM',
 'GIN',
 'GNB',
 'GUY',
 'HTI',
 'HND',
 'HKG',
 'HUN',
 'ISL',
 'IND',
 'IDN',
 'IRN',
 'IRL',
 'ISR',
 'ITA',
 'JAM',
 'JPN',
 'JOR',
 'KAZ',
 'KEN',
 'KOR',
 'KWT',
 'KGZ',
 'LAO',
 'LVA',
 'LBN',
 'LSO',
 'LBR',
 'LBY',
 'LTU',
 'LUX',
 'MKD',
 'MDG',
 'MWI',
 'MYS',
 'MDV',
 'MLI',
 'MRT',
 'MUS',
 'MEX',
 'MDA',
 'MNG',
 'MAR',
 'MOZ',
 'MMR',
 'NAM',
 'NPL',
 'NLD',
 'NCL',
 'NZL',
 'NIC',
 'NER',
 'NGA',
 'NOR',
 'PAK',
 'PAN',
 'PNG',
 'PRY',
 'PER',
 'PHL',
 'POL',
 'PRT',
 'ROU',
 'RUS',


In [25]:
df_filtrado = df2[df2["Country Code"].isin(iso_bien_lista)]

In [26]:
df_filtrado

,Country Name,Country Code,year,Bank nonperforming loans to total gross loans (%),Broad money (% of GDP),Current account balance (% of GDP),Deposit interest rate (%),Domestic credit to private sector (% of GDP),Domestic credit to private sector by banks (% of GDP),Exports of goods and services (% of GDP),...,"Inflation, consumer prices (annual %)",Lending interest rate (%),Net barter terms of trade index (2015 = 100),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Short-term debt (% of total external debt),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,Trade (% of GDP),"Unemployment, total (% of total labor force) (national estimate)"
165,Albania,ALB,1980,NaN,NaN,1.013876,NaN,NaN,NaN,23.957492,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.494093,5.600
166,Albania,ALB,1981,NaN,NaN,2.488694,NaN,NaN,NaN,23.810507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.100466,5.100
167,Albania,ALB,1982,NaN,NaN,-3.589153,NaN,NaN,NaN,20.072918,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.810562,3.600
168,Albania,ALB,1983,NaN,NaN,-2.035704,NaN,NaN,NaN,18.852075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.410668,4.400
169,Albania,ALB,1984,NaN,NaN,-1.512918,NaN,NaN,NaN,18.040331,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.115683,5.700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13922,Zimbabwe,ZWE,2020,NaN,10.002460,2.121236,4.518333,3.642132,3.607706,18.931225,...,557.201817,33.008333,107.1,51.329013,-80.610262,29.5755,18.680976,0.062756,40.178406,NaN
13923,Zimbabwe,ZWE,2021,NaN,10.239997,0.843388,8.059167,4.759522,4.759505,19.759617,...,98.546105,45.476667,96.6,88.552447,-30.318195,26.4874,9.228518,1.148220,44.114559,9.459
13924,Zimbabwe,ZWE,2022,NaN,12.673110,0.748243,34.720000,5.826323,5.826294,23.699220,...,104.705171,131.813333,94.0,374.954363,-38.093651,28.1510,6.084244,0.704319,55.067352,10.001
13925,Zimbabwe,ZWE,2023,NaN,11.204613,0.373211,62.820833,6.477697,6.477673,17.117076,...,NaN,170.293833,99.1,3509.172220,-68.877267,27.0548,19.516072,0.129141,40.291261,9.290


In [27]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8575 entries, 165 to 13926
Data columns (total 30 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Country Name                                                             8575 non-null   object 
 1   Country Code                                                             8575 non-null   object 
 2   year                                                                     8575 non-null   int64  
 3   Bank nonperforming loans to total gross loans (%)                        1865 non-null   float64
 4   Broad money (% of GDP)                                                   6577 non-null   float64
 5   Current account balance (% of GDP)                                       6585 non-null   float64
 6   Deposit interest rate (%)                                                4

In [28]:
df_filtrado["Country Name"].nunique()

164

In [29]:
df_codigo["country"].nunique()

165

In [30]:
df2

,Country Name,Country Code,year,Bank nonperforming loans to total gross loans (%),Broad money (% of GDP),Current account balance (% of GDP),Deposit interest rate (%),Domestic credit to private sector (% of GDP),Domestic credit to private sector by banks (% of GDP),Exports of goods and services (% of GDP),...,"Inflation, consumer prices (annual %)",Lending interest rate (%),Net barter terms of trade index (2015 = 100),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Short-term debt (% of total external debt),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,Trade (% of GDP),"Unemployment, total (% of total labor force) (national estimate)"
0,Afghanistan,AFG,1970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.692262,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,38.378709,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13922,Zimbabwe,ZWE,2020,NaN,10.002460,2.121236,4.518333,3.642132,3.607706,18.931225,...,557.201817,33.008333,107.1,51.329013,-80.610262,29.5755,18.680976,0.062756,40.178406,NaN
13923,Zimbabwe,ZWE,2021,NaN,10.239997,0.843388,8.059167,4.759522,4.759505,19.759617,...,98.546105,45.476667,96.6,88.552447,-30.318195,26.4874,9.228518,1.148220,44.114559,9.459
13924,Zimbabwe,ZWE,2022,NaN,12.673110,0.748243,34.720000,5.826323,5.826294,23.699220,...,104.705171,131.813333,94.0,374.954363,-38.093651,28.1510,6.084244,0.704319,55.067352,10.001
13925,Zimbabwe,ZWE,2023,NaN,11.204613,0.373211,62.820833,6.477697,6.477673,17.117076,...,NaN,170.293833,99.1,3509.172220,-68.877267,27.0548,19.516072,0.129141,40.291261,9.290


In [31]:
df_filtrado["Country Name"].nunique()

164

In [32]:
df_codigo["country"].nunique()

165

In [33]:
df_filtrado["Country Code"].nunique()

164

In [34]:
df_codigo["iso3"].nunique()

165

In [35]:
df_filtrado["Country Code"].unique()

array(['ALB', 'DZA', 'AGO', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BGD',
       'BRB', 'BLR', 'BEL', 'BLZ', 'BEN', 'BTN', 'BOL', 'BIH', 'BWA',
       'BRA', 'BRN', 'BGR', 'BFA', 'BDI', 'CPV', 'KHM', 'CMR', 'CAN',
       'CAF', 'TCD', 'CHL', 'CHN', 'COL', 'COM', 'COD', 'COG', 'CRI',
       'CIV', 'HRV', 'CYP', 'CZE', 'DNK', 'DJI', 'DMA', 'DOM', 'ECU',
       'EGY', 'SLV', 'GNQ', 'ERI', 'EST', 'SWZ', 'ETH', 'FJI', 'FIN',
       'FRA', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA', 'GRC', 'GRD', 'GTM',
       'GIN', 'GNB', 'GUY', 'HTI', 'HND', 'HKG', 'HUN', 'ISL', 'IND',
       'IDN', 'IRN', 'IRL', 'ISR', 'ITA', 'JAM', 'JPN', 'JOR', 'KAZ',
       'KEN', 'KOR', 'KWT', 'KGZ', 'LAO', 'LVA', 'LBN', 'LSO', 'LBR',
       'LBY', 'LTU', 'LUX', 'MDG', 'MWI', 'MYS', 'MDV', 'MLI', 'MRT',
       'MUS', 'MEX', 'MDA', 'MNG', 'MAR', 'MOZ', 'MMR', 'NAM', 'NPL',
       'NLD', 'NCL', 'NZL', 'NIC', 'NER', 'NGA', 'MKD', 'NOR', 'PAK',
       'PAN', 'PNG', 'PRY', 'PER', 'PHL', 'POL', 'PRT', 'ROU', 'RUS',
       'RWA', 'STP',

In [36]:
df_codigo["iso3"].unique() #FALTA YUGOSLAVIA en los datos originales

array(['ALB', 'DZA', 'AGO', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BGD',
       'BRB', 'BLR', 'BEL', 'BLZ', 'BEN', 'BTN', 'BOL', 'BIH', 'BWA',
       'BRA', 'BRN', 'BGR', 'BFA', 'BDI', 'KHM', 'CMR', 'CAN', 'CPV',
       'CAF', 'TCD', 'CHL', 'CHN', 'COL', 'COM', 'COD', 'COG', 'CRI',
       'CIV', 'HRV', 'CZE', 'CYP', 'DNK', 'DJI', 'DMA', 'DOM', 'ECU',
       'EGY', 'SLV', 'GNQ', 'ERI', 'EST', 'ETH', 'FJI', 'FIN', 'FRA',
       'GAB', 'GMB', 'GEO', 'DEU', 'GHA', 'GRC', 'GRD', 'GTM', 'GIN',
       'GNB', 'GUY', 'HTI', 'HND', 'HKG', 'HUN', 'ISL', 'IND', 'IDN',
       'IRN', 'IRL', 'ISR', 'ITA', 'JAM', 'JPN', 'JOR', 'KAZ', 'KEN',
       'KOR', 'KWT', 'KGZ', 'LAO', 'LVA', 'LBN', 'LSO', 'LBR', 'LBY',
       'LTU', 'LUX', 'MKD', 'MDG', 'MWI', 'MYS', 'MDV', 'MLI', 'MRT',
       'MUS', 'MEX', 'MDA', 'MNG', 'MAR', 'MOZ', 'MMR', 'NAM', 'NPL',
       'NLD', 'NCL', 'NZL', 'NIC', 'NER', 'NGA', 'NOR', 'PAK', 'PAN',
       'PNG', 'PRY', 'PER', 'PHL', 'POL', 'PRT', 'ROU', 'RUS', 'RWA',
       'KNA', 'STP',

In [37]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8575 entries, 165 to 13926
Data columns (total 30 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Country Name                                                             8575 non-null   object 
 1   Country Code                                                             8575 non-null   object 
 2   year                                                                     8575 non-null   int64  
 3   Bank nonperforming loans to total gross loans (%)                        1865 non-null   float64
 4   Broad money (% of GDP)                                                   6577 non-null   float64
 5   Current account balance (% of GDP)                                       6585 non-null   float64
 6   Deposit interest rate (%)                                                4

In [38]:
df_filtrado.loc[df_filtrado.year == 1970].info()

<class 'pandas.core.frame.DataFrame'>
Index: 139 entries, 210 to 13872
Data columns (total 30 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Country Name                                                             139 non-null    object 
 1   Country Code                                                             139 non-null    object 
 2   year                                                                     139 non-null    int64  
 3   Bank nonperforming loans to total gross loans (%)                        0 non-null      float64
 4   Broad money (% of GDP)                                                   87 non-null     float64
 5   Current account balance (% of GDP)                                       12 non-null     float64
 6   Deposit interest rate (%)                                                7 

In [39]:
# abajo el bucle para ver los infos por año

In [40]:
for y in sorted(df_filtrado["year"].unique()):
    print(f"AÑO DEL INFO!!!! -------------------------{y}-------------------------")
    print(df_filtrado[df_filtrado["year"] == y].info())

AÑO DEL INFO!!!! -------------------------1970-------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 139 entries, 210 to 13872
Data columns (total 30 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Country Name                                                             139 non-null    object 
 1   Country Code                                                             139 non-null    object 
 2   year                                                                     139 non-null    int64  
 3   Bank nonperforming loans to total gross loans (%)                        0 non-null      float64
 4   Broad money (% of GDP)                                                   87 non-null     float64
 5   Current account balance (% of GDP)                                       12 non-null     float64
 6   Dep

In [41]:
# abajo el bucle para ver los infos por pais

In [42]:
for y in sorted(df_filtrado["Country Name"].unique()):
    print(f"PAIS DEL INFO!!!! -------------------------{y}-------------------------")
    print(df_filtrado[df_filtrado["Country Name"] == y].info())

PAIS DEL INFO!!!! -------------------------Albania-------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 45 entries, 165 to 209
Data columns (total 30 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Country Name                                                             45 non-null     object 
 1   Country Code                                                             45 non-null     object 
 2   year                                                                     45 non-null     int64  
 3   Bank nonperforming loans to total gross loans (%)                        14 non-null     float64
 4   Broad money (% of GDP)                                                   30 non-null     float64
 5   Current account balance (% of GDP)                                       45 non-null     float64
 6   De

In [43]:
cols = df_filtrado.columns

In [44]:
df_paises_nulos = df_filtrado.groupby('Country Name').apply(lambda x : x.isna().mean().mean()).sort_values(ascending=False)
df_paises_nulos

### EXPLICACION ###
# GROUPBY - Me saca la tabla agrupada por paises
# APPLY(LAMBDA X:) - Tratame para cada X (en este caso pais) lo siguiente:
# x.isna().mean().mean() - En el pais x hazme la booleana de True/False en nulos, al hacer la media me lo da en numero, 
# 1º mean() me hace la media de nulos de por columna, 2º mean - me hace la media de nulos de todas las columnas

C:\Users\pacoc\AppData\Local\Temp\ipykernel_12160\1854304185.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_paises_nulos = df_filtrado.groupby('Country Name').apply(lambda x : x.isna().mean().mean()).sort_values(ascending=False)


Country Name
Eritrea              0.645455
New Caledonia        0.616970
Djibouti             0.544848
Equatorial Guinea    0.530909
Turkmenistan         0.501802
                       ...   
Peru                 0.086667
Thailand             0.080000
Kenya                0.076364
Egypt, Arab Rep.     0.070909
Colombia             0.069091
Length: 164, dtype: float64

In [45]:
paises_top = df_paises_nulos[df_paises_nulos <= 0.20].index.sort_values()
paises_top

Index(['Albania', 'Algeria', 'Argentina', 'Armenia', 'Australia', 'Azerbaijan',
       'Bangladesh', 'Belarus', 'Belize', 'Bolivia', 'Bosnia and Herzegovina',
       'Botswana', 'Brazil', 'Cameroon', 'Canada', 'Chile', 'China',
       'Colombia', 'Congo, Rep.', 'Czechia', 'Dominican Republic', 'Ecuador',
       'Egypt, Arab Rep.', 'El Salvador', 'Gabon', 'Gambia, The', 'Ghana',
       'Guatemala', 'Honduras', 'India', 'Indonesia', 'Jordan', 'Kazakhstan',
       'Kenya', 'Korea, Rep.', 'Kyrgyz Republic', 'Madagascar', 'Malaysia',
       'Mali', 'Mauritania', 'Mauritius', 'Mexico', 'Moldova', 'Mongolia',
       'Morocco', 'Nepal', 'Nicaragua', 'Niger', 'Pakistan', 'Paraguay',
       'Peru', 'Philippines', 'Russian Federation', 'Rwanda', 'Senegal',
       'Sierra Leone', 'Singapore', 'South Africa', 'Sri Lanka', 'Tanzania',
       'Thailand', 'Tunisia', 'Turkiye', 'Uganda', 'Ukraine', 'United Kingdom',
       'United States', 'Uruguay'],
      dtype='object', name='Country Name')

In [46]:
len(paises_top)

68

In [47]:
df_propuesta = df_filtrado.loc[df_filtrado['Country Name'].isin(paises_top),:]
df_propuesta

,Country Name,Country Code,year,Bank nonperforming loans to total gross loans (%),Broad money (% of GDP),Current account balance (% of GDP),Deposit interest rate (%),Domestic credit to private sector (% of GDP),Domestic credit to private sector by banks (% of GDP),Exports of goods and services (% of GDP),...,"Inflation, consumer prices (annual %)",Lending interest rate (%),Net barter terms of trade index (2015 = 100),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Short-term debt (% of total external debt),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,Trade (% of GDP),"Unemployment, total (% of total labor force) (national estimate)"
165,Albania,ALB,1980,NaN,NaN,1.013876,NaN,NaN,NaN,23.957492,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.494093,5.600
166,Albania,ALB,1981,NaN,NaN,2.488694,NaN,NaN,NaN,23.810507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.100466,5.100
167,Albania,ALB,1982,NaN,NaN,-3.589153,NaN,NaN,NaN,20.072918,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.810562,3.600
168,Albania,ALB,1983,NaN,NaN,-2.035704,NaN,NaN,NaN,18.852075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.410668,4.400
169,Albania,ALB,1984,NaN,NaN,-1.512918,NaN,NaN,NaN,18.040331,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.115683,5.700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13426,Uruguay,URY,2020,2.335771,59.126896,-0.641294,4.794616,27.884637,27.847128,26.188281,...,9.756406,11.231259,107.6,42.013292,0.458651,NaN,NaN,12.809403,47.731873,10.413
13427,Uruguay,URY,2021,1.326336,58.750620,-2.433511,3.799038,26.712977,26.678879,32.944605,...,7.747914,7.450648,103.9,43.554575,-3.262141,NaN,NaN,8.983196,58.315041,9.328
13428,Uruguay,URY,2022,1.386676,52.557440,-3.522450,6.235004,26.623834,26.608468,33.331613,...,9.104380,10.865362,97.3,41.171083,5.437763,NaN,NaN,6.549880,61.030795,7.877
13429,Uruguay,URY,2023,1.650773,51.123443,-3.044701,8.101361,28.577104,28.560787,28.140301,...,5.869104,11.895433,102.8,38.823917,8.202351,NaN,NaN,7.206551,52.910987,8.355


In [48]:
for y in sorted(df_propuesta["Country Name"].unique()):
    print(f"PAIS DEL INFO!!!! -------------------------{y}-------------------------")
    print(df_propuesta[df_propuesta["Country Name"] == y].info())

PAIS DEL INFO!!!! -------------------------Albania-------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 45 entries, 165 to 209
Data columns (total 30 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Country Name                                                             45 non-null     object 
 1   Country Code                                                             45 non-null     object 
 2   year                                                                     45 non-null     int64  
 3   Bank nonperforming loans to total gross loans (%)                        14 non-null     float64
 4   Broad money (% of GDP)                                                   30 non-null     float64
 5   Current account balance (% of GDP)                                       45 non-null     float64
 6   De

In [49]:
df_propuesta.groupby('Country Name').apply(lambda x : x.isna().mean().mean()).sort_values(ascending=False)

C:\Users\pacoc\AppData\Local\Temp\ipykernel_12160\482157568.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_propuesta.groupby('Country Name').apply(lambda x : x.isna().mean().mean()).sort_values(ascending=False)


Country Name
Nepal               0.200000
Tanzania            0.199394
Australia           0.196970
Mali                0.195758
Gabon               0.195152
                      ...   
Peru                0.086667
Thailand            0.080000
Kenya               0.076364
Egypt, Arab Rep.    0.070909
Colombia            0.069091
Length: 68, dtype: float64

In [50]:
df_propuesta.groupby('year').apply(lambda x : x.isna().mean().mean()).sort_values(ascending=False)

C:\Users\pacoc\AppData\Local\Temp\ipykernel_12160\1833923144.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_propuesta.groupby('year').apply(lambda x : x.isna().mean().mean()).sort_values(ascending=False)


year
2025    0.866667
1970    0.375000
1971    0.369048
1972    0.361905
1973    0.361310
1974    0.344643
1975    0.317262
1976    0.270833
1977    0.246429
1978    0.233333
1979    0.229762
1991    0.208824
1990    0.200000
2024    0.197549
1980    0.192398
1992    0.190196
1981    0.189080
1983    0.178161
1982    0.177586
1989    0.177049
1988    0.176111
1984    0.171839
1993    0.169118
1987    0.167797
1985    0.166667
1986    0.163218
1994    0.153431
1995    0.137255
1996    0.129412
1997    0.123529
1998    0.123529
1999    0.121078
2000    0.108824
2023    0.106863
2002    0.100980
2001    0.099020
2003    0.096569
2004    0.095588
2022    0.087745
2006    0.075980
2005    0.074510
2021    0.073529
2007    0.072549
2020    0.071569
2008    0.066176
2009    0.061275
2018    0.057843
2019    0.057843
2010    0.054902
2011    0.052941
2016    0.051961
2015    0.050490
2013    0.050490
2012    0.050000
2014    0.049510
2017    0.047059
dtype: float64

In [52]:
df_propuesta.to_excel('./pruebas_paco/df_top_68.xlsx')